# Task 3 di Kaggle: Retrieval (RQ3) + Stance (RQ4)
Kode ditarik dari **GitHub**, data besar dan checkpoint dari **Google Drive** (`gdown`), Komdigi dari **Kaggle Dataset**.
Kaggle tidak punya `drive.mount`, jadi Drive dipakai lewat link berbagi ("Anyone with the link").

Sebelum mulai:
1. Settings → Accelerator **GPU (T4/P100)**, Internet **On**.
2. Add Input → cari dataset `ireddragonicy/indonesian-hoax-news-dataset`.
3. (Repo privat saja) Add-ons → Secrets → `GITHUB_TOKEN`.
4. Kode task 3 harus sudah di-**push** ke GitHub. Saat ini masih uncommitted di laptop.

In [ ]:
# ===== KONFIGURASI: ubah di sini =====
GITHUB_REPO   = "danenftyessir/indonesian-hoax-checker"
GITHUB_BRANCH = "main"
GDRIVE_DATA_FOLDER_ID = ""   # folder Drive berisi data/checkpoint tambahan (opsional); isi ID dari URL folder
GDRIVE_OUT_FOLDER_ID  = ""   # tidak dipakai untuk upload (lihat sel terakhir); biarkan kosong
KAGGLE_INPUT_GLOB = "/kaggle/input/*/komdigi_hoaks.csv"
RUN_STAGE1_TRAINING = True   # fine-tune IndoBERT di IndoNLI (~GPU 20-40 menit)

In [ ]:
import os, subprocess, glob, shutil, pathlib
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    token = None
url = f"https://{token+'@' if token else ''}github.com/{GITHUB_REPO}.git"
WORK = "/kaggle/working"
REPO = f"{WORK}/indonesian-hoax-checker"
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(["git","clone","--depth","1","-b",GITHUB_BRANCH,url,REPO],check=True)
os.chdir(REPO)
!git log --oneline -3

In [ ]:
!pip -q install -e . "datasets>=2.18" "accelerate>=0.27" gdown rank_bm25 2>&1 | tail -3
import torch; print("cuda:", torch.cuda.is_available())

## Data
Komdigi dari Kaggle Input, IndoNLI dari GitHub raw, tambahan (BRIN, checkpoint, dll.) dari Drive.

In [ ]:
import yaml
os.makedirs("data/raw/refuted_claims", exist_ok=True)
hits = glob.glob(KAGGLE_INPUT_GLOB)
assert hits, "Dataset Komdigi belum ditambahkan sebagai Input"
shutil.copy(hits[0], "data/raw/refuted_claims/komdigi_hoaks.csv")

src = yaml.safe_load(open("configs/data_sources.yaml"))["indonli"]
os.makedirs(src["local_raw"], exist_ok=True)
for f in src["files"]:
    !wget -q -nc -P {src["local_raw"]} {src["base_url"]}{f}
!ls -la data/raw/refuted_claims data/raw/stance/indonli

In [ ]:
if GDRIVE_DATA_FOLDER_ID:
    import gdown
    gdown.download_folder(id=GDRIVE_DATA_FOLDER_ID, output="/kaggle/working/gdrive_in", quiet=False)
    # Salin ke struktur repo; sesuaikan nama jika isi folder Drive berbeda.
    # contoh: data/raw/stance/brin_fakenews_mafindo.csv, artifacts/checkpoints/stance_final/
    for rel in ["data", "artifacts"]:
        p = pathlib.Path("/kaggle/working/gdrive_in")/rel
        if p.exists(): shutil.copytree(p, rel, dirs_exist_ok=True)
else:
    print("GDRIVE_DATA_FOLDER_ID kosong, lewati.")

## Fase 1: klaim, split, konversi stance

In [ ]:
!python -m indonesian_hoax_checker.retrieval.corpus data/raw/refuted_claims/komdigi_hoaks.csv data/interim/refuted_claims/claims_raw.jsonl
!python -m indonesian_hoax_checker.evaluation.splits data/interim/refuted_claims/claims_raw.jsonl
for s in ["train","val","test_lay","test_expert"]:
    !python -m indonesian_hoax_checker.stance.dataset indonli data/raw/stance/indonli/{s}.jsonl data/processed/stance/indonli_{s}.jsonl --split-name {s}

## RQ3: retrieval (BM25 vs E5, tanpa/dengan filter entitas)

In [ ]:
Q = "data/annotations/claim_verification/retrieval_queries.jsonl"
runs = [("rq3_bm25_nofilter","configs/models/retrieval_bm25.yaml",""),
        ("rq3_bm25_entfilter","configs/models/retrieval_bm25.yaml","--entity-filter"),
        ("rq3_e5_nofilter","configs/models/retrieval_embeddings.yaml",""),
        ("rq3_e5_entfilter","configs/models/retrieval_embeddings.yaml","--entity-filter")]
for name,cfg,flag in runs:
    print("=== ",name)
    r = subprocess.run(f"python -m indonesian_hoax_checker.retrieval.evaluate {cfg} --queries {Q} --run-name {name} {flag}",shell=True)
    if r.returncode: print("GAGAL (cek: kueri belum dianotasi / claims_entities.jsonl belum ada)")

## RQ4: stance

In [ ]:
if RUN_STAGE1_TRAINING:
    !python -m indonesian_hoax_checker.stance.train configs/models/stance_nli.yaml         --train data/processed/stance/indonli_train.jsonl --dev data/processed/stance/indonli_val.jsonl         --output artifacts/checkpoints/stance_indonli --seed 42

In [ ]:
models = [("rq4_mdeberta_zeroshot","MoritzLaurer/mDeBERTa-v3-base-xnli-multilingual-nli-2mil7")]
if os.path.exists("artifacts/checkpoints/stance_indonli"): models.append(("rq4_indobert_indonli","artifacts/checkpoints/stance_indonli"))
if os.path.exists("artifacts/checkpoints/stance_final"):   models.append(("rq4_indobert_final","artifacts/checkpoints/stance_final"))
for split in ["indonli_test_expert","indonli_val"]:
    for name,m in models:
        !python -m indonesian_hoax_checker.stance.predict --model {m} --pairs data/processed/stance/{split}.jsonl --run-name {name}_{split}

In [ ]:
import json
for f in sorted(glob.glob("outputs/metrics/*.json")):
    print(f); print(json.dumps(json.load(open(f)),indent=1,ensure_ascii=False)[:800])

## Simpan hasil
Notebook tidak bisa menulis ke Drive tanpa kredensial. Dua cara:
- **Mudah:** zip di bawah, lalu klik tab *Output* → Download, unggah manual ke Drive.
- Checkpoint besar sebaiknya tidak di-commit; unggah ke Drive dan isi `GDRIVE_DATA_FOLDER_ID` untuk run berikutnya.

In [ ]:
!cd /kaggle/working && zip -rq task3_results.zip indonesian-hoax-checker/outputs indonesian-hoax-checker/artifacts/checkpoints indonesian-hoax-checker/data/splits indonesian-hoax-checker/data/processed
!ls -lh /kaggle/working/task3_results.zip